# Preprocessing

La seguente esercitazione è finalizzata a comprendere quali sono le librerie e i comandi di base più utili per eseguire pre-processing su un dataset.

Il dataset oggetto d'esame riporta l'unione di quattro studi clinici su pazienti con sindromi coronariche. Ogni record è classificato sulla base di quattro etichette che identificano diverse gravità della sindrome.

## Esercizio 0 : Importazione dei pacchetti e del dataset

In [1]:
import numpy as np
import pandas as pd # manipolazione dati e importazione(!!!)

In [2]:
# importazione dati
# NB: index_col = 0 dice che la prima colonna del dataset va usata come id per le osservazioni e non come feature
dd = pd.read_csv('data/heart_disease_uci.csv', sep = ';', index_col = 0)

In [3]:
# visualizzazione del dataset
dd

,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
id,,,,,,,,,,,,,,,
1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,54,Female,VA Long Beach,asymptomatic,127.0,333.0,True,st-t abnormality,154.0,False,0.0,NaN,NaN,NaN,1
917,62,Male,VA Long Beach,typical angina,NaN,139.0,False,st-t abnormality,NaN,NaN,NaN,NaN,NaN,NaN,0
918,55,Male,VA Long Beach,asymptomatic,122.0,223.0,True,st-t abnormality,100.0,False,0.0,NaN,NaN,fixed defect,2


In [4]:
# dimensioni dataset
dd.shape

(920, 15)

## Esercizio 1: Analisi delle colonne/righe eccessivamente sparse per la rimozione

Questo esercizio a primo impatto potrebbe richiedere poco sforzo. Basta calcolare il numero di NA per colonna/riga usando la funzione `X.isna().sum(axis = 0/1)`. In realtà un'attenta analisi dovrebbe portarci a valutare quale criterio applicare per decidere se una colonna/riga deve essere **eliminata** dal dataset. Possibili criteri possono essere:
- Elimina una colonna *se ha almeno il 50% di valori mancanti*;
- Elimina una riga *se ha più del 70% di valori mancanti*.

In questa fase del pre-processing non ci soffermeremo sull'**imputazione dei dati mancanti**, che affronteremo in un secondo momento.

In [5]:
# Numero di dati mancanti per colonna
dd.isna().sum() # questo comando restituisce il numero di valori mancanti per ogni colonna del dataset

age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64

In [6]:
# nota, axis = 0 indica che stiamo applicando la funzione colonna per colonna
dd.apply(lambda x: x.isna().sum(), axis = 0) # stesso comando, ma con apply, più lento!

age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64

In [7]:
# quali righe hanno più valori mancanti (Top 5)
dd.isna().sum(axis = 1)

id
1      0
2      0
3      0
4      0
5      0
      ..
916    3
917    7
918    2
919    7
920    3
Length: 920, dtype: int64

Con questi comandi abbiamo ottenuto un quadro abbastanza intuitivo della situazione: le feature `ca` e `thal` hanno il numero più elevato di valori mancanti. Mentre le righe 917 e 919 hanno 7 colonne in corrispondenza delle quali mancano dei valori. 
In realtà, un quadro più completo può essere ottenuto se riusciamo a ricavare le seguenti informazioni:
1. Tra le colonne, quali hanno la percentuale più alta di dati mancanti?
2. Tra le righe, quali (e quante) hanno la percentuale più alta di dati mancanti? 

### Rimozione colonne

In [8]:
# percentuale di valori mancanti per colonna (arrotondato a 2 decimali)
dd.isna().mean().round(4) * 100 

age          0.00
sex          0.00
dataset      0.00
cp           0.00
trestbps     6.41
chol         3.26
fbs          9.78
restecg      0.22
thalch       5.98
exang        5.98
oldpeak      6.74
slope       33.59
ca          66.41
thal        52.83
num          0.00
dtype: float64

In [9]:
# quali variabili hanno più del 50% di valori mancanti? Restituisci solo nomi colonne che soddisfano la condizione
dd.columns[dd.isna().mean() > 0.5] # ca e thal 

Index(['ca', 'thal'], dtype='object')

Le variabili `ca` e `thal` superano la soglia del 50% che avevamo imposto. Se la consegna fosse quella di rimuovere tali colonne, esse andrebbero rimosse. 

**NB: la rimozione di variabili è un'operazione che può alterare i dati originali. Sarebbe sempre meglio evitare che il dato grezzo venga manipolato, per cui è opportuno**

In [10]:
# rimozione delle colonne con più del 50% di valori mancanti in un nuovo dataframe
dd_clean = dd.drop(columns = dd.columns[dd.isna().mean() > 0.5])
dd_clean.shape # le colonne ora sono 13!

(920, 13)

### Rimozione righe

Per semplicità lavoreremo sul dataframe ripulito; il numero di missing per riga può essere diverso rispetto a prima se si considera che sono state rimosse due colonne. 

In [11]:
dd_clean.isna().mean(axis = 1).round(4)* 100

id
1       0.00
2       0.00
3       0.00
4       0.00
5       0.00
       ...  
916     7.69
917    38.46
918     7.69
919    38.46
920     7.69
Length: 920, dtype: float64

Questa visualizzazione continua ad essere poco informativa. Quali sono le osservazioni (righe) con il più alto numero di NA?

In [12]:
# quali righe hanno più valori mancanti? Top 5
dd_clean.isna().mean(axis = 1).round(4).sort_values(ascending = False).head() * 100

id
885    46.15
876    46.15
902    46.15
789    38.46
889    38.46
dtype: float64

Abbiamo finalmente introdotto il metodo `sort_values()`, in cui abbiamo specificato di NON volere l'ordine crescente; dopo aver calcolato le percentuali di NA per riga, le abbiamo ordinate.

In questo caso **nessuna riga supera la soglia critica imposta**. Ma se avessimo imposto tale soglia al 40%, allora avremmo dovuto rimuovere le unità 885, 876, 902. 

In [13]:
# rimozione righe con più del 40% di valori mancanti
dd_clean = dd_clean[dd_clean.isna().mean(axis = 1) <= 0.4]
dd_clean.shape # abbiamo 3 righe in meno!

(917, 13)

Il codice scritto sino ad ora non è pulito: spesso abbiamo ripetuto la stessa sintassi continuamente. Quale può essere una soluzione ottimale? 

Sfruttiamo la **programmazione funzionale**!

In [14]:
def dropsparse(df, max_na_col = 0.5, max_na_row = 0.5):
    '''
    Rimuove le colonne e le righe con più dati mancanti rispetto alla soglia impostata. 
    df: dataframe da pulire
    max_na_col: soglia per la rimozione delle colonne (default 0.5)
    max_na_row: soglia per la rimozione delle righe (default 0.5)
    
    Restituisce un dataframe pulito.
    '''
    # 1. Rimozione colonne
    df_clean = df.drop(columns = df.columns[df.isna().mean() > max_na_col])
    # 2. Rimozione righe
    df_clean = df_clean[df_clean.isna().mean(axis = 1) <= max_na_row]
    
    return df_clean



In [15]:
dd_clean_new = dropsparse(dd, max_na_col = 0.5, max_na_row = 0.4)
dd_clean_new.shape 

(917, 13)

## Esercizio 2: Encoding delle feature categoriche

L'encoding delle feature categoriche è una fase obbligatoria per task di machine learning. È essenzialmente il processo di trasformazione di etichette testuali (es. "Rosso", "Blu", "Giallo" o "Piccolo", "Medium", "Grande") in valori numerici.

I modelli di Machine Learning svolgono compiti come il *calcolo di distanze* e l'*ottimizzazione di pesi* che richiedono l'uso di soli valori numerici.

L'encoding di feature categoriche può essere suddiviso in 3:

- **Dati nominali** (no ordinamento): quando le modalità sono distinte e non seguono un ordine logico (città, colori, marche di auto) allora bisogna usare tecniche che le trattino come tali. Si usa il **One-Hot Encoder**, che genera un vettore binario sparso;

- **Dati ordinali** (ordinamento): quando le modalità seguono un ordine ben preciso e si vuole preservare quella gerarchia. Si usa l'**Ordinal Encoder**, che assegna numeri crescenti in base all'ordine delle feature;

- **Dati ad alta cardinalità** (troppe modalità): 

In [16]:
# Come prima cosa dobbiamo capire se i dati sono nominali o ordinali
# per farlo possiamo usare il metodo .unique() per vedere i valori unici di ogni colonna
dd_clean.apply(lambda x: x.unique())

age         [63, 67, 37, 41, 56, 62, 57, 53, 44, 52, 48, 5...
sex                                            [Male, Female]
dataset      [Cleveland, Hungary, Switzerland, VA Long Beach]
cp          [typical angina, asymptomatic, non-anginal, at...
trestbps    [145.0, 160.0, 120.0, 130.0, 140.0, 172.0, 150...
chol        [233.0, 286.0, 229.0, 250.0, 204.0, 236.0, 268...
fbs                                        [True, False, nan]
restecg       [lv hypertrophy, normal, st-t abnormality, nan]
thalch      [150.0, 108.0, 129.0, 187.0, 172.0, 178.0, 160...
exang                                      [False, True, nan]
oldpeak     [2.3, 1.5, 2.6, 3.5, 1.4, 0.8, 3.6, 0.6, 3.1, ...
slope                     [downsloping, flat, upsloping, nan]
num                                           [0, 2, 1, 3, 4]
dtype: object

Alcune delle variabili categoriali sono contraddistinte da `nan`, dati mancanti. Lo strumento che useremo per l'encoding **non accetta dati mancanti**, per cui bisognerà gestire questi valori mancanti *prima* di eseguire l'encoder:

1. Rimozione delle righe con missing (rischio di perdere molta informazione)

2. Imputazione semplice (es. "Ignoto", o con valore più frequente)

In ogni caso, per svolgere l'imputazione prima e l'encoding poi, bisogna introdurre il modulo `sklearn.preprocessing` di `scikit-learn`.

In [17]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer

# Nativamente, sklearn restituisce array numpy, ma possiamo configurarlo per restituire DataFrame
from sklearn._config import set_config 
set_config(transform_output= 'pandas') # per far sì che le trasformazioni restituiscano DataFrame

In [18]:
# Crea due Imputer distinti per dati categorici e numerici
imp_cat = SimpleImputer(strategy = 'most_frequent') # oppure strategy = 'constant', fill_value = 'missing'
imp_num = SimpleImputer(strategy = 'median') # oppure strategy = 'mean'

# Identifica colonne numeriche e categoriche
num_cols = dd_clean.select_dtypes(include = ['number']).columns
cat_cols = dd_clean.select_dtypes(exclude = ['number']).columns

# Imputazione dati mancanti
dd_imp = dd_clean.copy()

dd_imp[cat_cols] = imp_cat.fit_transform(dd_clean[cat_cols])
dd_imp[num_cols] = imp_num.fit_transform(dd_clean[num_cols])

dd_imp.isna().sum()


age         0
sex         0
dataset     0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalch      0
exang       0
oldpeak     0
slope       0
num         0
dtype: int64

In [19]:
dd_imp.dtypes 

age         float64
sex          object
dataset      object
cp           object
trestbps    float64
chol        float64
fbs          object
restecg      object
thalch      float64
exang        object
oldpeak     float64
slope        object
num         float64
dtype: object

Dobbiamo assegnare la strategia corretta alle colonne a cui vogliamo applicare l'encoder. Il tutto può essere automatizzato, ma è necessario che vengano specificate **quali variabili categoriali sono anche ordinali e qual è l'ordine delle modalità** che l'ordinal encoder deve rispettare.

In [20]:
# Nel nostro caso slope sembra essere l'unica variabile ordinale
ord_cols = ['slope']
slope_order = [['missing', 'downsloping', 'flat', 'upsloping']] # da meno a più grave

# Filtra le colonne nominali
nom_cols = [col for col in cat_cols if col not in ord_cols]

Invece di eseguire l'encoding colonna per colonna, Scikit-Learn consente di usare un tool chiamato `ColumnTransformer`, dal modulo `sklearn.compose`, che consente di applicare encoders diversi a gruppi di colonne in un unico passaggio.

In [21]:
from sklearn.compose import ColumnTransformer

In [22]:
preprocessor = ColumnTransformer(
    transformers=[
       
        ('nominal', OneHotEncoder(drop='if_binary', sparse_output=False), nom_cols),
        
        ('ordinal', OrdinalEncoder(categories=slope_order), ord_cols),
        
        ('numerical', 'passthrough', num_cols)
    ],
    remainder='passthrough' 
)
dd_enc = preprocessor.fit_transform(dd_imp)

In [23]:
# tipo colonne
dd_enc.apply(lambda x: x.unique())

nominal__sex_Male                                                           [1.0, 0.0]
nominal__dataset_Cleveland                                                  [1.0, 0.0]
nominal__dataset_Hungary                                                    [0.0, 1.0]
nominal__dataset_Switzerland                                                [0.0, 1.0]
nominal__dataset_VA Long Beach                                              [0.0, 1.0]
nominal__cp_asymptomatic                                                    [0.0, 1.0]
nominal__cp_atypical angina                                                 [0.0, 1.0]
nominal__cp_non-anginal                                                     [0.0, 1.0]
nominal__cp_typical angina                                                  [1.0, 0.0]
nominal__fbs_True                                                           [1.0, 0.0]
nominal__restecg_lv hypertrophy                                             [1.0, 0.0]
nominal__restecg_normal                    

L'encoding è stato eseguito, adesso possiamo procedere.

# Riepilogo 

## 0. Importazione e visualizzazione dati

Abbiamo imparato che per questa fase è indispensabile lavorare con `pandas` per l'importazione dei dati con `pd.read_csv('percorso/dati.csv')`, specificando eventualmente il separatore se non è la virgola di default, e se leggere la prima colonna come un id.
- `X.head()` per visualizzare le prime righe e colonne del dataframe;
- `X.shape()` per visualizzare il numero di (righe, colonne).

## 1. Analisi dei dati mancanti (riga e colonna)

- `X.isna().mean(axis = 0/1)` per vedere la proporzione di NA per colonna/riga;

- Aggiungendo il metodo `.sort_values(ascending = False)` possiamo ordinarle per numero di missing

- La rimozione delle colonne sfrutta la sintassi `X.drop(columns = X.columns[X.isna().mean >= 0.5])`;

- La rimozione delle righe sfrutta la sintassi `X[X.isna().mean(axis = 1) <= 0.5]`, tecnicamente in questo caso stiamo *conservando le righe con al più tot% NA*.